# 04 — Models

Does a uniform statewide swing predict 2026 from 2021, and does knowing
anything about a seat beat it?

Run `python3 -m src.features` first if `data/model/seat_features.csv` isn't
there yet.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.features import load_seat_features
from src.models import run_comparison, statewide_swing, uniform_swing_prediction, target

seats = load_seat_features()
print(f"{len(seats)} seats, both years polled")
seats.head(3)

## The swing

TMC lost 7.4 points of vote share and BJP gained 7.8. The two-party swing is
half of what changed hands between them, so about 7.6 points.

In [ ]:
swing = statewide_swing(seats)
print(f"two-party swing to BJP: {swing:.2f} points")
print(f"that moves a two-party margin by {2*swing:.2f} points")

## Baseline

Apply that one number to all 293 seats and recount. No candidate, no
incumbency, no local anything.

It has to clear a lower bar first: 207 of 293 seats went BJP, so saying "BJP"
every time is already right 70.6% of the time.

In [ ]:
y = target(seats)
pred = uniform_swing_prediction(seats, swing)
print(f"baseline calls {pred.sum()} BJP seats, actual was {y.sum()}")
print(f"seats correct: {(pred == y).sum()} of {len(seats)}  ({(pred == y).mean()*100:.1f}%)")

## Everything against everything

Four fitted models, all scored out-of-fold over 5 folds so no seat is graded
by a model that trained on it.

The `pre` and `post` split matters. `pre` uses only what was knowable before
polling day. `post` adds the 2026 roll change and turnout change, which are
measured at the same time as the result — a model using those is explaining,
not predicting.

In [ ]:
table = run_comparison(seats)
table

Two things worth noticing.

Uniform swing does most of the work. It takes you from 70.6% to 84.0% knowing
nothing about any seat. The best model adds about three more points on top of
that, which is real but much smaller than the gap the baseline closes.

And the `post` features make things **worse**, not better. The electoral roll
shrinking by 5 million names was the headline of the descriptive write-up, but
once you know a seat's 2021 result, knowing how its roll moved doesn't help
you call 2026. Whatever moved these seats, that isn't it.

In [ ]:
table.to_csv("../results/model_comparison.csv", index=False)